## Tratamiento de datos faltantes

In [ ]:
# Cargamos los paquetes necesarios ACORDARNOS DE HACER EL AMBIENTE VIRTUAL
import matplotlib.pyplot as plt 
import polars as pl
import seaborn as sns
import numpy as np
import pyprojroot
from plotnine import *
import pandas as pd

In [ ]:
# Definir la ruta raiz del proyecto
ROOT = pyprojroot.here()

# Cargamos los datos
datos_entrenamiento = pl.read_parquet(ROOT / "datos" / "temporada1.parquet")

In [ ]:
# Categorias consideradas swing
categorias_swing = [
    "foul",
    "hit_into_play",
    "swinging_strike",
    "swinging_strike_blocked",
    "foul_tip",
    "foul_bunt",
    "missed_bunt",
    "bunt_foul_tip",
    "foul_pitchout"
]

# Creamos la variable dicotómica
datos_entrenamiento = datos_entrenamiento.with_columns(
    pl.when(pl.col("description").is_in(categorias_swing))
    .then(1)
    .otherwise(0)
    .alias("swing")
)

datos_entrenamiento = datos_entrenamiento.with_columns(
    (pl.col("sz_top") - pl.col("sz_bot")).alias("altura_zona")
)

In [ ]:
na = (
    datos_entrenamiento
    .null_count()
    .transpose(include_header=True)
    .filter(pl.col("column_0") > 0)
)

na

Se realizó un análisis de los valores faltantes presentes en el conjunto de datos con el objetivo de determinar la estrategia de tratamiento más adecuada. Del total de 709.852 observaciones, se identificaron 5.131 registros con al menos un valor ausente, lo que representa aproximadamente un 0,72% del conjunto de datos.

In [ ]:


datos_con_na = datos_entrenamiento.filter(
    pl.any_horizontal(pl.all().is_null())
)

datos_con_na.height

In [ ]:

datos_entrenamiento.with_columns(
    pl.sum_horizontal(pl.all().is_null()).alias("NA_por_fila")
).filter(
    pl.col("NA_por_fila") > 0
).group_by("NA_por_fila").len()

Al analizar la distribución de valores faltantes por observación, se observa que la mayoría de los registros incompletos presentan una cantidad reducida de valores ausentes: 4.712 observaciones contienen únicamente un dato faltante y 52 observaciones presentan dos valores faltantes. Sin embargo, también se identifican 367 registros con entre 9 y 10 variables ausentes simultáneamente, evidenciando una pérdida importante de información en dichas observaciones.

In [ ]:
datos_entrenamiento.with_columns(
    pl.sum_horizontal(pl.all().is_null()).alias("NA_por_fila")
).filter(
    pl.col("NA_por_fila") >= 9
)

Los valores faltantes se concentran principalmente en variables asociadas a las características del lanzamiento, como la velocidad (release_speed), el tipo de lanzamiento (`pitch_type`), el movimiento de la pelota (`pfx_x` y `pfx_z`) y la ubicación del lanzamiento respecto de la zona de bateo (`plate_x`, `plate_z`, `sz_top`, `sz_bot` y `altura_zona`). Estas variables representan mediciones específicas del evento y resultan relevantes para explicar la decisión del bateador de realizar o no un swing.

Debido a que la proporción de observaciones incompletas es baja y que una parte de los registros presenta una pérdida significativa de información, se decide eliminar las observaciones que contienen valores faltantes. La imputación no resulta conveniente en este contexto, ya que podría introducir valores artificiales en variables que representan mediciones físicas del lanzamiento, modificando las relaciones reales entre las variables predictoras y la variable respuesta.

Luego de este procedimiento, se conserva más del 99% de las observaciones originales, manteniendo un volumen de datos suficiente para el entrenamiento y evaluación de los modelos predictivos.